# Setup

In [1]:
!python -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1


In [2]:
!pip install pandas scikit-learn tqdm transformers torch datasets pyarrow 'accelerate>=0.26.0' openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 36.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 55.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 36.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 2.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 60.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 kB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 116.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29/29 [datasets]/29 [datasets]e]s]ub]


In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch

# IRP Geral

## Treinamento do Modelo

In [ ]:
df_final = pd.read_parquet('./discursos_irp.parquet')

In [ ]:
df_final = df_final[df_final['labels'] != 9].reset_index(drop=True)

In [ ]:
def dividir_em_blocos(texto, tamanho_bloco=512):
    palavras = texto.split()
    return [" ".join(palavras[i:i + tamanho_bloco]) for i in range(0, len(palavras), tamanho_bloco)]

def expandir_dataset(df, coluna_texto="texto", coluna_label="label"):
    novos_textos = []
    novos_labels = []

    for _, row in df.iterrows():
        blocos = dividir_em_blocos(row[coluna_texto])
        novos_textos.extend(blocos)
        novos_labels.extend([row[coluna_label]] * len(blocos))

    return pd.DataFrame({coluna_texto: novos_textos, coluna_label: novos_labels})

df_final_blocos = expandir_dataset(df_final)

In [ ]:
df_train, df_val = train_test_split(df_final_blocos, test_size=0.2, stratify=df_final_blocos["labels"], random_state=42)

In [ ]:
# Passo 1: garantir rótulos corretos
df_train = df_train[df_train["labels"].isin([0, 1])].copy()
df_val = df_val[df_val["labels"].isin([0, 1])].copy()
df_train["labels"] = df_train["labels"].astype(int)
df_val["labels"] = df_val["labels"].astype(int)

# Passo 2: criar datasets
dataset = DatasetDict({
                        "train": Dataset.from_pandas(df_train),
                        "validation": Dataset.from_pandas(df_val)
                    })

In [ ]:
# Passo 3: tokenização + inclusão dos labels corretamente
model_name = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    encoding = tokenizer(examples["texto"], truncation=True, padding="max_length", max_length=512)
    encoding["labels"] = examples["labels"]
    return encoding

dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

# Passo 4: carregar modelo de classificação binária
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/32257 [00:00<?, ? examples/s]

Map:   0%|          | 0/8065 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Passo 5: definir métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Passo 6: argumentos de treinamento
training_args = TrainingArguments(
                                    output_dir="./irp_train_2.1",
                                    learning_rate=2e-5,
                                    per_device_train_batch_size=32,
                                    per_device_eval_batch_size=32,
                                    num_train_epochs=6,
                                    weight_decay=0.01,
                                    warmup_ratio=0.1,
                                    label_smoothing_factor=0.05
                                )

In [10]:
# Passo 7: treinar
trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=dataset["train"],
                    eval_dataset=dataset["validation"],
                    tokenizer=tokenizer,
                    compute_metrics=compute_metrics
                )

/tmp/ipykernel_131/929800995.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()
trainer.save_model("./irp_train_2.1")
tokenizer.save_pretrained("./irp_train_2.1")

Step,Training Loss
500,0.473000
1000,0.232500
1500,0.199500
2000,0.189500
2500,0.165400
3000,0.160400
3500,0.146900
4000,0.142800
4500,0.132600
5000,0.130300


('./irp_train_2.1/tokenizer_config.json',
 './irp_train_2.1/special_tokens_map.json',
 './irp_train_2.1/vocab.txt',
 './irp_train_2.1/added_tokens.json',
 './irp_train_2.1/tokenizer.json')

## Gerar o score para todo o dataset

In [5]:
# Caminho para o modelo treinado
model_path = "./irp"

# Carregar tokenizador e modelo
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Criar pipeline com GPU (device=0 para usar cuda:0)
classifier = pipeline(
                        task="text-classification",
                        model=model,
                        tokenizer=tokenizer,
                        device=0,               # Usa GPU se disponível
                        truncation=True,        # Garante que blocos longos sejam cortados corretamente
                    )

Device set to use cuda:0


In [6]:
df = pd.read_parquet('discursos_irp.parquet')

In [8]:
from tqdm.notebook import tqdm

def score_em_lotes_continuo(df, texto_col="texto", bloco_caracteres=1000, batch_size_pipeline=8, batch_size_textos=32):
    scores = []
    total = len(df)

    for i in tqdm(range(0, total, batch_size_textos), desc="Processando discursos"):
        batch_df = df.iloc[i:i+batch_size_textos]
        blocos_por_texto = []

        for texto in batch_df[texto_col]:
            blocos = [texto[i:i+bloco_caracteres] for i in range(0, len(texto), bloco_caracteres)]
            blocos_por_texto.append(blocos)

        blocos_flat = [bloco for blocos in blocos_por_texto for bloco in blocos]

        # Retorna todas as probabilidades para todas as classes
        resultados = classifier(blocos_flat, batch_size=batch_size_pipeline, truncation=True, return_all_scores=True)

        idx = 0
        for blocos in blocos_por_texto:
            probs_label1 = []

            for _ in blocos:
                prob_label_1 = resultados[idx][1]["score"]  # índice 1 = classe LABEL_1
                probs_label1.append(prob_label_1)
                idx += 1

            if probs_label1:
                media = sum(probs_label1) / len(probs_label1)
                scores.append(round(media * 100, 4))
            else:
                scores.append(None)

    return scores


In [9]:
df["score_populismo_cont"] = score_em_lotes_continuo(df)

Processando discursos:   0%|          | 0/1398 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [13]:
df2 = df.drop(columns=['labels', 'texto'])

# IRP-E

## Treinamento do Modelo

In [4]:
df_final = pd.read_parquet('./discursos_irp_v2.parquet')

In [5]:
df_final['labels'] = df_final['FLAG_POPULISMO_FINAL_IRPE']

In [6]:
df_final = df_final[df_final['labels'] != 9].reset_index(drop=True)

In [7]:
def dividir_em_blocos(texto, tamanho_bloco=512):
    palavras = texto.split()
    return [" ".join(palavras[i:i + tamanho_bloco]) for i in range(0, len(palavras), tamanho_bloco)]

def expandir_dataset(df, coluna_texto="texto", coluna_label="labels"):
    novos_textos = []
    novos_labels = []

    for _, row in df.iterrows():
        blocos = dividir_em_blocos(row[coluna_texto])
        novos_textos.extend(blocos)
        novos_labels.extend([row[coluna_label]] * len(blocos))

    return pd.DataFrame({coluna_texto: novos_textos, coluna_label: novos_labels})

df_final_blocos = expandir_dataset(df_final)

In [8]:
df_train, df_val = train_test_split(df_final_blocos, test_size=0.2, stratify=df_final_blocos["labels"], random_state=42)

In [9]:
# Passo 1: garantir rótulos corretos
df_train = df_train[df_train["labels"].isin([0, 1])].copy()
df_val = df_val[df_val["labels"].isin([0, 1])].copy()
df_train["labels"] = df_train["labels"].astype(int)
df_val["labels"] = df_val["labels"].astype(int)

# Passo 2: criar datasets
dataset = DatasetDict({
                        "train": Dataset.from_pandas(df_train),
                        "validation": Dataset.from_pandas(df_val)
                    })

In [10]:
# Passo 3: tokenização + inclusão dos labels corretamente
model_name = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    encoding = tokenizer(examples["texto"], truncation=True, padding="max_length", max_length=512)
    encoding["labels"] = examples["labels"]
    return encoding

dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

# Passo 4: carregar modelo de classificação binária
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/89989 [00:00<?, ? examples/s]

Map:   0%|          | 0/22498 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Passo 5: definir métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [12]:
# Passo 6: argumentos de treinamento
training_args = TrainingArguments(
                                    output_dir="./irp-e",
                                    learning_rate=2e-5,
                                    per_device_train_batch_size=32,
                                    per_device_eval_batch_size=32,
                                    num_train_epochs=6,
                                    weight_decay=0.01,
                                    warmup_ratio=0.1,
                                    label_smoothing_factor=0.05
                                )

In [13]:
# Passo 7: treinar
trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=dataset["train"],
                    eval_dataset=dataset["validation"],
                    tokenizer=tokenizer,
                    compute_metrics=compute_metrics
                )

/tmp/ipykernel_769/929800995.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train("./irp-e/checkpoint-12500")
trainer.save_model("./irp-e")
tokenizer.save_pretrained("./irp-e")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Step,Training Loss
13000,0.130000
13500,0.126600
14000,0.126900
14500,0.126700
15000,0.123600
15500,0.123700
16000,0.124000
16500,0.122500


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packa

('./irp-e/tokenizer_config.json',
 './irp-e/special_tokens_map.json',
 './irp-e/vocab.txt',
 './irp-e/added_tokens.json',
 './irp-e/tokenizer.json')

In [16]:
# Caminho para o modelo treinado
model_path = "./irp-e"

# Carregar tokenizador e modelo
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Criar pipeline com GPU (device=0 para usar cuda:0)
classifier = pipeline(
                        task="text-classification",
                        model=model,
                        tokenizer=tokenizer,
                        device=0,               # Usa GPU se disponível
                        truncation=True,        # Garante que blocos longos sejam cortados corretamente
                    )

Device set to use cuda:0


In [17]:
df = pd.read_parquet('discursos_irp_v2.parquet')

In [18]:
from tqdm.notebook import tqdm

def score_em_lotes_continuo(df, texto_col="texto", bloco_caracteres=1000, batch_size_pipeline=8, batch_size_textos=32):
    scores = []
    total = len(df)

    for i in tqdm(range(0, total, batch_size_textos), desc="Processando discursos"):
        batch_df = df.iloc[i:i+batch_size_textos]
        blocos_por_texto = []

        for texto in batch_df[texto_col]:
            blocos = [texto[i:i+bloco_caracteres] for i in range(0, len(texto), bloco_caracteres)]
            blocos_por_texto.append(blocos)

        blocos_flat = [bloco for blocos in blocos_por_texto for bloco in blocos]

        # Retorna todas as probabilidades para todas as classes
        resultados = classifier(blocos_flat, batch_size=batch_size_pipeline, truncation=True, return_all_scores=True)

        idx = 0
        for blocos in blocos_por_texto:
            probs_label1 = []

            for _ in blocos:
                prob_label_1 = resultados[idx][1]["score"]  # índice 1 = classe LABEL_1
                probs_label1.append(prob_label_1)
                idx += 1

            if probs_label1:
                media = sum(probs_label1) / len(probs_label1)
                scores.append(round(media * 100, 4))
            else:
                scores.append(None)

    return scores


In [19]:
df["score_irpe"] = score_em_lotes_continuo(df)

Processando discursos:   0%|          | 0/1398 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [25]:
df2 = df.drop(columns=['FLAG_POPULISMO_FINAL_IRPE', 'texto'])

In [26]:
df2.to_excel('irpe.xlsx')

# IRP-R

## Treinamento do Modelo

In [2]:
df_final = pd.read_parquet('./discursos_irp_v2.parquet')

In [3]:
df_final['labels'] = df_final['FLAG_POPULISMO_FINAL_IRPR']

In [4]:
df_final = df_final[df_final['labels'] != 9].reset_index(drop=True)

In [5]:
def dividir_em_blocos(texto, tamanho_bloco=512):
    palavras = texto.split()
    return [" ".join(palavras[i:i + tamanho_bloco]) for i in range(0, len(palavras), tamanho_bloco)]

def expandir_dataset(df, coluna_texto="texto", coluna_label="labels"):
    novos_textos = []
    novos_labels = []

    for _, row in df.iterrows():
        blocos = dividir_em_blocos(row[coluna_texto])
        novos_textos.extend(blocos)
        novos_labels.extend([row[coluna_label]] * len(blocos))

    return pd.DataFrame({coluna_texto: novos_textos, coluna_label: novos_labels})

df_final_blocos = expandir_dataset(df_final)

In [6]:
df_train, df_val = train_test_split(df_final_blocos, test_size=0.2, stratify=df_final_blocos["labels"], random_state=42)

In [7]:
# Passo 1: garantir rótulos corretos
df_train = df_train[df_train["labels"].isin([0, 1])].copy()
df_val = df_val[df_val["labels"].isin([0, 1])].copy()
df_train["labels"] = df_train["labels"].astype(int)
df_val["labels"] = df_val["labels"].astype(int)

# Passo 2: criar datasets
dataset = DatasetDict({
                        "train": Dataset.from_pandas(df_train),
                        "validation": Dataset.from_pandas(df_val)
                    })

In [8]:
# Passo 3: tokenização + inclusão dos labels corretamente
model_name = "google-bert/bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    encoding = tokenizer(examples["texto"], truncation=True, padding="max_length", max_length=512)
    encoding["labels"] = examples["labels"]
    return encoding

dataset = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

# Passo 4: carregar modelo de classificação binária
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Map:   0%|          | 0/164004 [00:00<?, ? examples/s]

Map:   0%|          | 0/41002 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Passo 5: definir métricas
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [10]:
# Passo 6: argumentos de treinamento
training_args = TrainingArguments(
                                    output_dir="./irp-r",
                                    learning_rate=2e-5,
                                    per_device_train_batch_size=64,
                                    per_device_eval_batch_size=64,
                                    num_train_epochs=6,
                                    weight_decay=0.01,
                                    warmup_ratio=0.1,
                                    label_smoothing_factor=0.05,
                                    fp16=True,
                                    dataloader_num_workers=8
                                )

In [11]:
# Passo 7: treinar
trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=dataset["train"],
                    eval_dataset=dataset["validation"],
                    tokenizer=tokenizer,
                    compute_metrics=compute_metrics
                )

/tmp/ipykernel_2381/929800995.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
trainer.train()
trainer.save_model("./irp-r")
tokenizer.save_pretrained("./irp-r")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss
500,0.522100
1000,0.263100
1500,0.226400
2000,0.204400
2500,0.190000
3000,0.175900
3500,0.175500
4000,0.170600
4500,0.167600
5000,0.166200


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packa

('./irp-r/tokenizer_config.json',
 './irp-r/special_tokens_map.json',
 './irp-r/vocab.txt',
 './irp-r/added_tokens.json',
 './irp-r/tokenizer.json')

In [13]:
# Caminho para o modelo treinado
model_path = "./irp-r"

# Carregar tokenizador e modelo
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Criar pipeline com GPU (device=0 para usar cuda:0)
classifier = pipeline(
                        task="text-classification",
                        model=model,
                        tokenizer=tokenizer,
                        device=0,               # Usa GPU se disponível
                        truncation=True,        # Garante que blocos longos sejam cortados corretamente
                    )

Device set to use cuda:0


In [14]:
df = pd.read_parquet('discursos_irp_v2.parquet')

In [15]:
from tqdm.notebook import tqdm

def score_em_lotes_continuo(df, texto_col="texto", bloco_caracteres=1000, batch_size_pipeline=8, batch_size_textos=32):
    scores = []
    total = len(df)

    for i in tqdm(range(0, total, batch_size_textos), desc="Processando discursos"):
        batch_df = df.iloc[i:i+batch_size_textos]
        blocos_por_texto = []

        for texto in batch_df[texto_col]:
            blocos = [texto[i:i+bloco_caracteres] for i in range(0, len(texto), bloco_caracteres)]
            blocos_por_texto.append(blocos)

        blocos_flat = [bloco for blocos in blocos_por_texto for bloco in blocos]

        # Retorna todas as probabilidades para todas as classes
        resultados = classifier(blocos_flat, batch_size=batch_size_pipeline, truncation=True, return_all_scores=True)

        idx = 0
        for blocos in blocos_por_texto:
            probs_label1 = []

            for _ in blocos:
                prob_label_1 = resultados[idx][1]["score"]  # índice 1 = classe LABEL_1
                probs_label1.append(prob_label_1)
                idx += 1

            if probs_label1:
                media = sum(probs_label1) / len(probs_label1)
                scores.append(round(media * 100, 4))
            else:
                scores.append(None)

    return scores


In [16]:
df["score_irpr"] = score_em_lotes_continuo(df)

Processando discursos:   0%|          | 0/1398 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [17]:
df2 = df.drop(columns='texto')

In [18]:
df2.to_excel('irpr.xlsx')